In [1]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Preprocess Dataset

Convert the raw dataset to T5 text-to-text format (JSONL).

In [2]:
from src.t5_pipeline.preprocess import preprocess_dataset

dataset_dir = Path('dataset')
output_dir = Path('dataset/preprocessed_t5')

# Run preprocessing (only needs to be done once)
results = preprocess_dataset(dataset_dir, output_dir)
print(f"\nPreprocessing results: {results}")

Preprocessed train: 13084 examples, 0 dropped
Preprocessed valid: 700 examples, 0 dropped
Preprocessed test: 700 examples, 0 dropped

Preprocessing results: {'train': (13084, 0), 'valid': (700, 0), 'test': (700, 0)}


## 2. Load Preprocessed Data

In [3]:
from src.t5_pipeline.dataset import T5JointDataset, collate_fn

preprocessed_dir = Path('dataset/preprocessed_t5')

train_dataset = T5JointDataset(preprocessed_dir / 'train.jsonl')
val_dataset = T5JointDataset(preprocessed_dir / 'valid.jsonl')
test_dataset = T5JointDataset(preprocessed_dir / 'test.jsonl')

print(f"\nTrain: {len(train_dataset)} examples")
print(f"Valid: {len(val_dataset)} examples")
print(f"Test: {len(test_dataset)} examples")

Loaded 13084 examples from dataset/preprocessed_t5/train.jsonl
Loaded 700 examples from dataset/preprocessed_t5/valid.jsonl
Loaded 700 examples from dataset/preprocessed_t5/test.jsonl

Train: 13084 examples
Valid: 700 examples
Test: 700 examples


In [4]:
# Inspect a few examples
print("Example from training data:")
for i in range(3):
    example = train_dataset[i]
    print(f"\nInput:  {example['input_text']}")
    print(f"Target: {example['target_text']}")

Example from training data:

Input:  intent and slots: Add Don and Sherri to my Meditate to Sounds of Nature playlist
Target: intent: AddToPlaylist slots: entity_name=Don and Sherri playlist=Meditate to Sounds of Nature playlist_owner=my

Input:  intent and slots: put United Abominations onto my rare groove playlist
Target: intent: AddToPlaylist slots: entity_name=United Abominations playlist=rare groove playlist_owner=my

Input:  intent and slots: add the tune by misato watanabe to the Trapeo playlist
Target: intent: AddToPlaylist slots: artist=misato watanabe music_item=tune playlist=Trapeo


## 3. Initialize Model

In [5]:
from src.models.t5 import T5JointModel

# Use t5-small for faster training, t5-base for better performance
model = T5JointModel(model_name='t5-small')
model = model.to(device)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cuda
Loading T5 model 't5-small'... This may take a moment on first run.
T5 model 't5-small' loaded successfully!

Model parameters: 60,506,624


## 4. Setup Training

In [6]:
from src.t5_pipeline.train import get_optimizer

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
EPOCHS = 15
MAX_INPUT_LENGTH = 64
MAX_TARGET_LENGTH = 64

# Create dataloaders
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    collate_fn=collate_fn
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    collate_fn=collate_fn
)

# Optimizer
optimizer = get_optimizer(model, lr=LEARNING_RATE)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

Training batches: 818
Validation batches: 44


## 5. Training Loop

In [ ]:
from src.t5_pipeline.evaluate import evaluate, print_metrics

best_exact_match = 0.0
best_model_state = None

for epoch in range(EPOCHS):
    # Training
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in progress_bar:
        # Tokenize inside training loop
        input_encoding = model.tokenize_input(batch['input_text'], max_length=MAX_INPUT_LENGTH)
        labels = model.tokenize_target(batch['target_text'], max_length=MAX_TARGET_LENGTH)
        
        input_ids = input_encoding['input_ids'].to(device)
        attention_mask = input_encoding['attention_mask'].to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} - Average Training Loss: {avg_train_loss:.4f}")
    
    # Validation
    model.eval()
    predictions = []
    references = []
    
    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]"):
            input_encoding = model.tokenize_input(batch['input_text'], max_length=MAX_INPUT_LENGTH)
            input_ids = input_encoding['input_ids'].to(device)
            attention_mask = input_encoding['attention_mask'].to(device)
            
            # Generate predictions
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4
            )
            
            decoded_preds = model.decode(output_ids)
            predictions.extend(decoded_preds)
            references.extend(batch['target_text'])
    
    # Evaluate
    metrics = evaluate(predictions, references)
    print_metrics(metrics)
    
    # Save best model by exact match
    if metrics['exact_match'] > best_exact_match:
        best_exact_match = metrics['exact_match']
        best_model_state = model.model.state_dict().copy()
        print(f"New best model! Exact Match: {best_exact_match:.4f}")

Epoch 1/15 [Train]: 100%|██████████| 818/818 [02:25<00:00,  5.62it/s, loss=0.4721]



Epoch 1 - Average Training Loss: 1.3672


Epoch 1/15 [Valid]: 100%|██████████| 44/44 [00:16<00:00,  2.73it/s]



EVALUATION RESULTS
Intent Accuracy:     0.8457
Slot Precision:      0.5586
Slot Recall:         0.4746
Slot F1:             0.5132
Exact Match:         0.1543
Parse Failure Rate:  0.0000
Total Examples:      700
New best model! Exact Match: 0.1543


Epoch 2/15 [Train]: 100%|██████████| 818/818 [02:25<00:00,  5.61it/s, loss=0.1672]


...

Epoch 11/15 [Train]: 100%|██████████| 818/818 [02:25<00:00,  5.63it/s, loss=0.1018]



Epoch 11 - Average Training Loss: 0.0903


Epoch 11/15 [Valid]: 100%|██████████| 44/44 [00:17<00:00,  2.57it/s]



EVALUATION RESULTS
Intent Accuracy:     0.9800
Slot Precision:      0.9078
Slot Recall:         0.9014
Slot F1:             0.9046
Exact Match:         0.7657
Parse Failure Rate:  0.0000
Total Examples:      700
New best model! Exact Match: 0.7657


Epoch 12/15 [Train]:  18%|█▊        | 145/818 [00:26<02:02,  5.49it/s, loss=0.1192]

## 6. Save Best Model

In [ ]:
from src.t5_pipeline.train import save_model

# Load best weights back into model
if best_model_state is not None:
    model.model.load_state_dict(best_model_state)

# Save the model
save_path = Path('src/weights/t5_joint_model')
save_model(model, save_path)
print(f"Best model saved with Exact Match: {best_exact_match:.4f}")

Model saved to src/weights/t5_joint_model
Best model saved with Exact Match: 0.6686


## 7. Final Evaluation on Test Set

In [ ]:
test_dataloader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    collate_fn=collate_fn
)

model.eval()
predictions = []
references = []

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing"):
        input_encoding = model.tokenize_input(batch['input_text'], max_length=MAX_INPUT_LENGTH)
        input_ids = input_encoding['input_ids'].to(device)
        attention_mask = input_encoding['attention_mask'].to(device)
        
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_TARGET_LENGTH,
            num_beams=4
        )
        
        decoded_preds = model.decode(output_ids)
        predictions.extend(decoded_preds)
        references.extend(batch['target_text'])

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
test_metrics = evaluate(predictions, references)
print_metrics(test_metrics)

Testing: 100%|██████████| 44/44 [00:16<00:00,  2.66it/s]


TEST SET RESULTS

EVALUATION RESULTS
Intent Accuracy:     0.9700
Slot Precision:      0.8635
Slot Recall:         0.8428
Slot F1:             0.8530
Exact Match:         0.6657
Parse Failure Rate:  0.0000
Total Examples:      700


## 8. Inference Demo

In [ ]:
from src.t5_pipeline.inference import T5Inferencer

# Create inferencer with the trained model
inferencer = T5Inferencer(model=model, device=str(device))

# Test utterances
test_utterances = [
    "book a flight from delhi to mumbai",
    "play some jazz music",
    "what's the weather like in new york",
    "add this song to my favorites playlist"
]

print("\nInference Examples:")
print("="*60)
for utterance in test_utterances:
    result = inferencer.predict(utterance)
    print(f"\nUtterance: {utterance}")
    print(f"Intent:    {result['intent']}")
    print(f"Slots:     {result['slots']}")
    print(f"Raw:       {result['raw_output']}")


Inference Examples:

Utterance: book a flight from delhi to mumbai
Intent:    BookRestaurant
Slots:     {'city': 'Delhi', 'country': 'Mambai'}
Raw:       intent: BookRestaurant slots: city=Delhi country=Mambai

Utterance: play some jazz music
Intent:    PlayMusic
Slots:     {'genre': 'jazz'}
Raw:       intent: PlayMusic slots: genre=jazz

Utterance: what's the weather like in new york
Intent:    GetWeather
Slots:     {'city': 'new'}
Raw:       intent: GetWeather slots: city=new york

Utterance: add this song to my favorites playlist
Intent:    AddToPlaylist
Slots:     {'music_item': 'song', 'playlist': 'favourites', 'playlist_owner': 'my'}
Raw:       intent: AddToPlaylist slots: music_item=song playlist=favourites playlist_owner=my
